In [0]:
from pyspark.sql import functions as F

batch_id = "batch_001"

controls = (
    spark.table("workspace.migration_dev.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(controls) != 1:
    raise RuntimeError("Expected one batch control record")

control = controls[0]

latest = (
    spark.table("workspace.migration_dev.downstream_reconciliation")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("reconciled_at").desc())
    .limit(1)
    .collect()
)

if (
    control["batch_status"] != "APPROVED"
    or control["failed_checks"] != 0
    or not latest
    or latest[0]["load_completion_status"] != "COMPLETE"
    or latest[0]["validation_run_id"] != control["validation_run_id"]
):
    raise RuntimeError("Current batch is not approved and reconciled")

gold = (
    spark.table("workspace.migration_dev.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
)

archive_files = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(f"/Volumes/workspace/migration_dev/archive/{batch_id}")
    .select(
        F.regexp_extract("path", r"([^/]+)$", 1).alias("file_name"),
        F.regexp_replace("path", r"^dbfs:", "").alias("archive_path"),
        F.sha2("content", 256).alias("archive_sha256"),
        F.col("length").alias("archive_size"),
    )
)

documents = gold.join(archive_files, "file_name", "inner")

expected = control["expected_document_count"]

if (
    gold.count() != expected
    or archive_files.count() != expected
    or documents.count() != expected
    or documents.select("document_id").distinct().count() != expected
):
    raise RuntimeError("Archive count or document identity mismatch")

invalid = documents.filter(
    ~F.col("source_sha256").eqNullSafe(F.col("archive_sha256"))
    | ~F.col("file_size_bytes").eqNullSafe(F.col("archive_size"))
    | ~F.col("validation_run_id").eqNullSafe(control["validation_run_id"])
)

if invalid.limit(1).count() > 0:
    raise RuntimeError("Archive integrity or approval mismatch")

registry = documents.select(
    "batch_id",
    "document_id",
    "account_number",
    "document_type",
    "statement_date",
    "archive_path",
    F.col("archive_sha256").alias("document_sha256"),
    "file_size_bytes",
    "validation_run_id",
    F.lit(latest[0]["reconciliation_run_id"]).alias("reconciliation_run_id"),
    F.current_timestamp().alias("archived_at"),
    F.add_months(F.current_date(), 120).alias("retention_until"),
    F.lit(False).alias("legal_hold"),
)

registry.createOrReplaceTempView("current_registry_batch")

# Block conflicts with an existing registered document.
conflicts = spark.sql("""
    SELECT COUNT(*) AS conflicts
    FROM workspace.migration_dev.document_registry t
    JOIN current_registry_batch s
      ON t.batch_id = s.batch_id AND t.document_id = s.document_id
    WHERE NOT (t.document_sha256 <=> s.document_sha256)
       OR NOT (t.archive_path <=> s.archive_path)
       OR NOT (t.account_number <=> s.account_number)
       OR NOT (t.document_type <=> s.document_type)
       OR NOT (t.statement_date <=> s.statement_date)
       OR NOT (t.file_size_bytes <=> s.file_size_bytes)
""").first()["conflicts"]

if conflicts:
    raise RuntimeError("Existing registry records conflict with this batch")

# Preserve existing records, including retention dates and legal holds.
spark.sql("""
    MERGE INTO workspace.migration_dev.document_registry AS target
    USING current_registry_batch AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN NOT MATCHED THEN INSERT *
""")

saved = (
    spark.table("workspace.migration_dev.document_registry")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Registered documents: {saved.count()}")
display(saved.orderBy("document_id").limit(5))

Registered documents: 100


batch_id,document_id,account_number,document_type,statement_date,archive_path,document_sha256,file_size_bytes,validation_run_id,reconciliation_run_id,archived_at,retention_until,legal_hold
batch_001,DOC000001,TEST00000001,ACCOUNT_STATEMENT,2026-09-30,/Volumes/workspace/migration_dev/archive/batch_001/DOC000001.pdf,12298c6adcbc1f0ffed046f3c14ab064da327b4fab519c9e25014e6a8182c431,1542,4e66d857-a136-4b69-af88-a9acada6bd83,d87e1a20-bf1f-43c8-8341-1f3ee2a6f82a,2026-10-14T11:01:02.007Z,2036-10-14,false
batch_001,DOC000002,TEST00000002,ACCOUNT_STATEMENT,2026-09-30,/Volumes/workspace/migration_dev/archive/batch_001/DOC000002.pdf,e45b766bfe7a3f38c7303239ffab722e244474ccb300ef84529b48197cbb83ce,1542,4e66d857-a136-4b69-af88-a9acada6bd83,d87e1a20-bf1f-43c8-8341-1f3ee2a6f82a,2026-10-14T11:01:02.007Z,2036-10-14,false
batch_001,DOC000003,TEST00000003,ACCOUNT_STATEMENT,2026-09-30,/Volumes/workspace/migration_dev/archive/batch_001/DOC000003.pdf,eecd9ffaf648015b1dcc0a49067b9724e9a38b609a1174a19046bf6d33a662c7,1543,4e66d857-a136-4b69-af88-a9acada6bd83,d87e1a20-bf1f-43c8-8341-1f3ee2a6f82a,2026-10-14T11:01:02.007Z,2036-10-14,false
batch_001,DOC000004,TEST00000004,ACCOUNT_STATEMENT,2026-09-30,/Volumes/workspace/migration_dev/archive/batch_001/DOC000004.pdf,8a1b437face5c2d7feb3fbee111a1b98b3f72afe81110e92fa55279fc7e3ea76,1542,4e66d857-a136-4b69-af88-a9acada6bd83,d87e1a20-bf1f-43c8-8341-1f3ee2a6f82a,2026-10-14T11:01:02.007Z,2036-10-14,false
batch_001,DOC000005,TEST00000005,ACCOUNT_STATEMENT,2026-09-30,/Volumes/workspace/migration_dev/archive/batch_001/DOC000005.pdf,e9907b9c7b5e0b6d0836ae782f5a5de81780eb22dbba4802102a97ed69773eb6,1542,4e66d857-a136-4b69-af88-a9acada6bd83,d87e1a20-bf1f-43c8-8341-1f3ee2a6f82a,2026-10-14T11:01:02.007Z,2036-10-14,false
